In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [3]:
access_key = os.getenv("Access_key_ID")
secret_key = os.getenv("Secret_access_key")
bucket = os.getenv("BUCKET_NAME")
region = os.getenv("REGION_NAME")

In [4]:
spark = SparkSession.builder \
    .appName("S3DataTransformation") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", secret_key) \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.region", region) \
    .getOrCreate()

25/04/17 09:42:19 WARN Utils: Your hostname, brempong-HP-EliteBook-840-G7-Notebook-PC resolves to a loopback address: 127.0.1.1; using 192.168.36.43 instead (on interface wlp0s20f3)
25/04/17 09:42:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/brempong/Ecommerce-DataLakehouse/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/brempong/.ivy2/cache
The jars for the packages stored in: /home/brempong/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3d3cb501-5353-4943-89b5-25a7433cc7ce;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.1 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.901 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 260ms :: artifacts dl 12ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.1 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	----------------------------

In [5]:
spark

In [6]:
folder_path = f"s3a://{bucket}/raw-data/products/"
df = spark.read.csv(folder_path, header=True, inferSchema=True)
df.show()

25/04/17 09:42:48 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+----------+-------------+-----------+-------------------+
|product_id|department_id| department|       product_name|
+----------+-------------+-----------+-------------------+
|         1|            4|      Books|    Product_1_Store|
|         2|            2|      Books|    Product_2_There|
|         3|            4|      Books|     Product_3_Hand|
|         4|            6|     Sports|       Product_4_Tv|
|         5|            1|       Toys|     Product_5_Easy|
|         6|            1|     Sports|    Product_6_Woman|
|         7|            5|       Home|     Product_7_Yard|
|         8|            5|      Books|  Product_8_Manager|
|         9|            6|       Toys|     Product_9_Face|
|        10|            4|     Sports|   Product_10_Sport|
|        11|            2|   Clothing|  Product_11_Parent|
|        12|            2|      Books| Product_12_Contain|
|        13|            4|     Sports|Product_13_Audience|
|        14|            1|       Home|     Product_14_Jo

In [8]:
df.select("department").distinct().show()

+-----------+
| department|
+-----------+
|       Home|
|     Sports|
|Electronics|
|   Clothing|
|      Books|
|       Toys|
+-----------+



In [9]:
partioned_product = df.repartition(6, col("department"))
partioned_product.show()

+----------+-------------+----------+--------------------+
|product_id|department_id|department|        product_name|
+----------+-------------+----------+--------------------+
|         1|            4|     Books|     Product_1_Store|
|         2|            2|     Books|     Product_2_There|
|         3|            4|     Books|      Product_3_Hand|
|         7|            5|      Home|      Product_7_Yard|
|         8|            5|     Books|   Product_8_Manager|
|        12|            2|     Books|  Product_12_Contain|
|        14|            1|      Home|      Product_14_Job|
|        26|            1|      Home|Product_26_Develo...|
|        29|            4|      Home|    Product_29_Cause|
|        30|            1|      Home| Product_30_Thousand|
|        31|            3|     Books|       Product_31_My|
|        32|            6|      Home| Product_32_Together|
|        33|            6|     Books|     Product_33_Card|
|        43|            6|     Books|Product_43_Confer..